# 🚀 Welcome to Day 14.

You're now entering a phase that many ML projects never reach:

Build model
↓
Serve model
↓
Monitor model

Up to Day 13 we focused on:

Recommendation Quality
+
API Development
+
Caching

Now we focus on:

Observability
Reliability
Operations

which are critical in production systems.


---

🚀 Day 14 — Logging, Monitoring & Cache Expiration (TTL)

🎯 Goal

Improve:

✅ Debugging

✅ Monitoring

✅ Cache Management

✅ Production Readiness


---

Problem 1 — Print statements don't scale

Right now:

print(
    f"Cache hit: {cache_key}"
)

works.

But imagine:

1000 requests/minute

You'll get:

Cache hit...
Cache hit...
Response time...
Error...

mixed together.

Hard to debug.


---

🚀 Part 1 — Proper Logging


---

Step 1

Import logging

import logging


---

Step 2

Configure logger

Near top of app.py:

logging.basicConfig(

    filename='logs/api.log',

    level=logging.INFO,

    format=
    '%(asctime)s - %(levelname)s - %(message)s'
)


---

Create logs folder

Project structure:

recommendation_system/

logs/
    api.log


---

Step 3

Replace print statements

Instead of:

print(
    f"Cache hit: {cache_key}"
)

Use:

logging.info(
    f"Cache hit: {cache_key}"
)


---

Instead of:

print(
    f"Response time: {time}"
)

Use:

logging.info(
    f"Response time: {time}"
)


---

Errors:

logging.error(
    str(e)
)


---

Example log file

2026-05-30 09:00:01 INFO Cache hit: 1_10

2026-05-30 09:00:04 INFO Response time: 0.42 sec

2026-05-30 09:00:10 ERROR Invalid user


---

🧠 Why logging matters

Production debugging rarely uses:

print()

Most systems use:

Log files
↓
Monitoring tools
↓
Alerting systems


---

🚀 Part 2 — Cache Expiration (TTL)

Current problem:

recommendation_cache={}

Cache grows forever.


---

Example:

User 1
User 2
...
User 100000

Cache size:

100000 entries

Memory keeps growing.


---

What is TTL?

TTL:

Time To Live

Example:

Store recommendation
↓
Keep for 5 minutes
↓
Automatically expire


---

Step 1

Store timestamp with cache

Instead of:

recommendation_cache[
    cache_key
] = response

Store:

recommendation_cache[
    cache_key
] = {

    "data":response,

    "timestamp":
        time.time()
}


---

Step 2

Define TTL

Near top:

CACHE_TTL = 300

Meaning:

300 sec
=
5 min


---

Step 3

Check cache validity

Replace cache lookup:

if cache_key in recommendation_cache:

with:

if cache_key in recommendation_cache:

    cached_item = (
        recommendation_cache[
            cache_key
        ]
    )

    age = (
        time.time()
        -
        cached_item[
            "timestamp"
        ]
    )

    if age < CACHE_TTL:

        logging.info(
            f"Cache hit: {cache_key}"
        )

        return cached_item[
            "data"
        ]


---

What happens now?

Request
↓
Stored
↓
5 minutes
↓
Cache valid

After:

5 minutes
↓
Recompute recommendation
↓
Update cache


---

🚀 Part 3 — Cache Cleanup Endpoint

Let's see expired entries.


---

Create:

@app.get(
    "/cache-cleanup"
)

def cleanup_cache():


---

Logic:

current_time = time.time()

keys_to_remove = []

for key,value in recommendation_cache.items():

    age = (
        current_time
        -
        value[
            "timestamp"
        ]
    )

    if age > CACHE_TTL:

        keys_to_remove.append(
            key
        )


---

Remove:

for key in keys_to_remove:

    del recommendation_cache[
        key
    ]


---

Return:

return {

    "removed":
    len(keys_to_remove),

    "remaining":
    len(
        recommendation_cache
    )
}


---

Example

{
  "removed":15,
  "remaining":8
}


---

🚀 Part 4 — Enhanced Metrics

Current metrics:

{
  "requests":45
}

Let's improve.


---

Add globals:

request_count = 0

cache_hits = 0

error_count = 0


---

Cache hit:

cache_hits += 1


---

Exception:

error_count += 1


---

Metrics endpoint:

@app.get(
    "/metrics"
)

def metrics():

    return {

        "total_requests":
            request_count,

        "cache_hits":
            cache_hits,

        "errors":
            error_count,

        "cache_entries":
            len(
                recommendation_cache
            )
    }


---

Example:

{
  "total_requests":100,

  "cache_hits":65,

  "errors":2,

  "cache_entries":25
}


---

🧠 Important Concepts Learned Today

1. Logging

print()
↓
logging


---

2. Observability

Ability to understand:

What happened?
When?
Why?


---

3. TTL Caching

Cache
↓
Expiration
↓
Refresh


---

4. Cache Lifecycle Management

Prevent memory growth.


---

5. Service Metrics

Track:

Traffic
Performance
Errors
Cache efficiency


---

🎯 Homework

1

Generate recommendations.

Check:

logs/api.log

Does logging work?


---

2

Test cache hit counter.

Verify:

/metrics

changes.


---

3

Set:

CACHE_TTL = 10

temporarily.

Wait 10 seconds.

Check:

/cache-cleanup

Observe expired cache removal.


---

After Day 14

We will be very close to deployment territory:

Day 15
↓
Docker 🐳

which you've already requested as a mandatory project milestone before we consider the recommendation system complete. 🚀

